# 14 · Kerala roads and local-body boundaries — OpenStreetMap (Geofabrik)

**Source:** OpenStreetMap contributors, Geofabrik "India – Southern Zone" extract (updated daily).
**Licence:** ODbL 1.0 — "© OpenStreetMap contributors".

1. **Roads:** every way tagged `highway=*` (national/state highways down to residential roads, tracks and
   footpaths) in Kerala plus about 20 km around it, with `oneway`, `maxspeed`, `surface`, `bridge`, `ferry`.
   Used for road travel time from fire stations and hospitals, and walking distance to schools (plan D5).
2. **Local bodies:** grama panchayats, municipalities and municipal corporations, mapped by the OSM Kerala
   community as `boundary=local_authority` with `local_authority:IN=*` (OSM wiki marks Kerala complete:
   942 rural and 94 urban bodies). Used to report results by panchayat.

The 0.56 GB zone file is downloaded to a temporary folder (Colab: `/content/tmp`) and deleted after the two
Kerala extracts are saved in `data/raw/osm/`.

## Imports and settings

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))
import os
if "postgis" in os.environ.get("PROJ_LIB", "").lower():  # an old PROJ database from a PostGIS install breaks rasterio
    os.environ.pop("PROJ_LIB")
import datetime as dt
import json
import re
import shutil
import time

import geopandas as gpd
import pyogrio
import requests
import urllib3.util.connection
from shapely.geometry import shape

urllib3.util.connection.HAS_IPV6 = False  # Geofabrik over IPv6 timed out from this network (2026-11-07); IPv4 works

OUT_DIR = PROJECT / "data" / "raw" / "osm"
TMP_DIR = (Path("/content/tmp") if IN_COLAB else PROJECT / "data" / "tmp") / "geofabrik"  # Colab: fast local disk, not Drive
URL = "https://download.geofabrik.de/asia/india/southern-zone-latest.osm.pbf"
ADM1 = PROJECT / "data" / "raw" / "boundaries" / "geoBoundaries-IND-ADM1.geojson"
BBOX = (74.6, 7.9, 77.6, 13.1)  # Kerala plus about 20 km around it (west, south, east, north)
DATE = dt.date.today().isoformat()
HEADERS = {"User-Agent": "msc-spatial-inequality-study/0.1 (academic project)"}

In [2]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio", "rasterio"], check=True)

### `download()`

In [3]:
def download(url, f, min_free_gb=5):
    """Download once. A finished file keeps its name; an interrupted one stays *.part and is fetched again."""
    f.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(4):  # servers can be slow to answer; wait and retry
        try:
            size = int(requests.head(url, headers=HEADERS, allow_redirects=True, timeout=300).headers.get("content-length", 0))
            break
        except requests.RequestException as e:
            print("  server slow, retrying:", type(e).__name__)
            time.sleep(60)
    else:
        raise SystemExit(f"no answer from {url}")
    if f.exists() and (size == 0 or f.stat().st_size == size):
        print(f"already saved: {f.name} ({f.stat().st_size / 1e6:.0f} MB)")
        return f
    free = shutil.disk_usage(f.parent).free
    if free - size < min_free_gb * 1e9:
        raise SystemExit(f"not enough disk for {f.name}: {size / 1e9:.2f} GB needed, {free / 1e9:.1f} GB free")
    part = f.with_name(f.name + ".part")
    with requests.get(url, headers=HEADERS, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(part, "wb") as out:
            shutil.copyfileobj(r.raw, out, length=1 << 20)
    if size and part.stat().st_size != size:
        raise SystemExit(f"{f.name}: got {part.stat().st_size} bytes, expected {size}")
    part.replace(f)
    print(f"downloaded: {f.name} ({size / 1e6:.0f} MB)")
    return f

### `tag()`

In [4]:
def tag(other_tags, key):
    """Value of one key in GDAL's other_tags text ('"key"=>"value",...')."""
    if not isinstance(other_tags, str):  # missing tags come back as NaN
        return None
    m = re.search(r'"' + re.escape(key) + r'"=>"((?:[^"\\]|\\.)*)"', other_tags)
    return m.group(1) if m else None

### `extract_roads()`

In [5]:
def extract_roads(pbf):
    """Every OSM way with a highway=* tag in the Kerala box (vehicle roads and footpaths)."""
    r = pyogrio.read_dataframe(pbf, layer="lines", columns=["osm_id", "name", "highway", "other_tags"],
                               where="highway IS NOT NULL", bbox=BBOX)
    for key in ["oneway", "maxspeed", "surface", "bridge", "ferry"]:
        r[key] = [tag(t, key) for t in r["other_tags"]]
    return r.drop(columns="other_tags")

### `extract_local_bodies()`

In [6]:
def extract_local_bodies(pbf):
    """Kerala grama panchayats, municipalities and corporations: boundary=local_authority (OSM Kerala community)."""
    p = pyogrio.read_dataframe(pbf, layer="multipolygons", columns=["osm_id", "osm_way_id", "name", "boundary",
                                                                     "admin_level", "other_tags"],
                               where="boundary = 'local_authority'", bbox=BBOX)
    p["local_authority_IN"] = [tag(t, "local_authority:IN") for t in p["other_tags"]]
    p["name_en"] = [tag(t, "name:en") for t in p["other_tags"]]
    p["name_ml"] = [tag(t, "name:ml") for t in p["other_tags"]]
    kerala = [shape(f["geometry"]) for f in json.loads(ADM1.read_text(encoding="utf-8"))["features"]
              if f["properties"]["shapeName"] == "Kerala"][0]
    inside = p.geometry.representative_point().within(kerala.buffer(0.01))  # drop neighbouring states' bodies
    return p[inside].drop(columns="other_tags")

### `run_all()`

In [7]:
def run_all():
    roads_f, lsg_f = OUT_DIR / f"kerala_roads_{DATE}.gpkg", OUT_DIR / f"kerala_local_bodies_{DATE}.gpkg"
    if list(OUT_DIR.glob("kerala_roads_*.gpkg")) and list(OUT_DIR.glob("kerala_local_bodies_*.gpkg")):
        print("already saved:", [f.name for f in OUT_DIR.glob("kerala_*.gpkg")])
        return
    pbf = download(URL, TMP_DIR / "southern-zone-latest.osm.pbf")
    roads = extract_roads(pbf)
    pyogrio.write_dataframe(roads, roads_f)
    print(f"roads: {len(roads):,} ways -> {roads_f.name} ({roads_f.stat().st_size / 1e6:.0f} MB)")
    lsg = extract_local_bodies(pbf)
    pyogrio.write_dataframe(lsg, lsg_f)
    print(f"local bodies: {len(lsg):,} -> {lsg_f.name}")
    pbf.unlink()  # keep only the Kerala extracts, to save disk

## Run

In [8]:
run_all()

already saved: ['kerala_local_bodies_2026-11-07.gpkg', 'kerala_roads_2026-11-07.gpkg']


## Check

In [9]:
roads = pyogrio.read_dataframe(sorted(OUT_DIR.glob("kerala_roads_*.gpkg"))[-1])
print("roads by class (top 15):")
print(roads["highway"].value_counts().head(15).to_string())
lsg = pyogrio.read_dataframe(sorted(OUT_DIR.glob("kerala_local_bodies_*.gpkg"))[-1])
print("\nboundaries by OSM admin_level: 8 = local bodies, 7 = block panchayats, 9/10 = wards (partly mapped)")
print(lsg.groupby(["admin_level", "local_authority_IN"], dropna=False).size().to_string())
lb = lsg[lsg["admin_level"].eq("8")]
print(f"\nlocal bodies (admin_level 8): {len(lb)} -> {lb['local_authority_IN'].value_counts().to_dict()}")
print("official (Kerala LSG): 941 grama panchayats + 87 municipalities + 6 corporations = 1,034")
print("geometry valid:", int(lb.geometry.is_valid.sum()), "of", len(lb))

roads by class (top 15):
highway
residential      530233
unclassified     201129
service           57889
tertiary          34584
track             24397
secondary         20732
path              15566
living_street     15047
footway           14193
primary            9766
trunk              7842
steps              2045
trunk_link         1303
construction        952
road                884



boundaries by OSM admin_level: 8 = local bodies, 7 = block panchayats, 9/10 = wards (partly mapped)
admin_level  local_authority_IN   
10           gram_panchayat           10368
             grama_panchayath             1
             municipal_corporation      256
             municipality              1838
             ward                        20
             NaN                          1
7            block_panchayat            152
8            gram_panchayat             936
             municipal_corporation        6
             municipality                85
9            ward                         2

local bodies (admin_level 8): 1027 -> {'gram_panchayat': 936, 'municipality': 85, 'municipal_corporation': 6}
official (Kerala LSG): 941 grama panchayats + 87 municipalities + 6 corporations = 1,034
geometry valid: 1027 of 1027
